This notebook focuses on Natural Language Processing (NLP) sentiment analysis of tweets to predict whether they are related to a disaster or not. The notebook uses various data cleaning techniques to preprocess the data and then employs the TF-IDF vectorizer to convert the tweets into numerical form. Several popular ML models such as XGBoost, AdaBoost, Logistic Regression, Naive Bayes and Random Forest are used to classify the tweets into disaster-related or not. The notebook aims to compare the performance of these models and identify the most accurate one for this task. Overall, this notebook presents a comprehensive approach to sentiment analysis of tweets with the goal of predicting disaster-related tweets.

# Importing Libraries

In [ ]:
import nltk
import numpy as np
import pandas as pd
import re
import seaborn as sns
import gensim
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import string
stop_words=nltk.corpus.stopwords.words('english')

from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression


stopwords=nltk.corpus.stopwords.words('english')
ps=nltk.PorterStemmer()

# Reading Data

In [ ]:
train = pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')
test = pd.read_csv('/kaggle/input/nlp-getting-started/test.csv')

In [ ]:
print('shape of training set: ', train.shape)
print('shape of testing set: ', test.shape)

In [ ]:
train.columns

In [ ]:
train.info()

In [ ]:
train = train.drop(columns=['keyword', 'location'])
train.head()

In [ ]:
train.isna().sum()

In [ ]:
sns.countplot(x='target', data=train)
plt.title('Target Distribution');

print(train['target'].value_counts())

In [ ]:
def clean_text(text):
    text= "".join ([word for word in text if word not in string.punctuation])
    tokens= re.split('\W+',text)
    text= [word for word in tokens if word not in stopwords]
    return text

train['text']= train['text'].apply(lambda x: clean_text(x.lower()))

train.head()

In [ ]:
type(train['text'])
train.columns

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")

In [ ]:
train['text']= train['text'].astype(str)

In [ ]:
def transform(text):
    doc=nlp(text,disable=['parser','ner'])
    lemmas=[token.lemma_ for token in doc]
    a_lemmas=[lemma for lemma in lemmas if lemma.isalpha()]
    return ' '.join(a_lemmas)

train['text']=train['text'].apply(transform)
train['text']

In [ ]:
train.tail()

# TF-IDF Vectorizer

In [ ]:
#Using TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

tfidf_vect=TfidfVectorizer(analyzer=clean_text)
X_tfidf= tfidf_vect.fit_transform(train['text'])
print(X_tfidf.shape)
print(tfidf_vect.get_feature_names_out())

In [ ]:
X_tfidf_df=pd.DataFrame(X_tfidf.toarray())
X_tfidf_df.columns=tfidf_vect.get_feature_names_out()

X_tfidf_df

In [ ]:
y = train['target']

In [ ]:
#splitting into training and testing for tf-idf
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_tfidf_df, y, test_size=0.20, random_state=42)


# XGBoost

In [ ]:
import xgboost as xgb

xgb_class = xgb.XGBClassifier()
xgb_class.fit(X_train,y_train)
pred = xgb_class.predict(X_test)

acc = accuracy_score(y_test,pred)
print(f'Accuracy of model is',acc*100)

recall = recall_score(y_test,pred)
print(f'Recall of model is',recall*100)

prec = precision_score(y_test,pred)
print(f'Precision of model is',prec*100)

f1 = f1_score(y_test,pred)
print(f'F1 Score of model is', f1*100)

# AdaBoost

In [ ]:
from sklearn.ensemble import AdaBoostClassifier 

In [ ]:
ada = AdaBoostClassifier(n_estimators=100, random_state=0)
ada.fit(X_train, y_train) 
preds = ada.predict(X_test)

acc = accuracy_score(y_test,preds)
print(f'Accuracy of model is',acc*100)

recall = recall_score(y_test,preds)
print(f'Recall of model is',recall*100)

prec = precision_score(y_test,preds)
print(f'Precision of model is',prec*100)

f1 = f1_score(y_test,preds)
print(f'F1 Score of model is', f1*100)

# Logistic Regression

In [ ]:
log = LogisticRegression()
log.fit(X_train,y_train)
log_pred = log.predict(X_test)

acc = accuracy_score(y_test,log_pred)
print(f'Accuracy of model is',acc*100)

recall = recall_score(y_test,log_pred)
print(f'Recall of model is',recall*100)

prec = precision_score(y_test,log_pred)
print(f'Precision of model is',prec*100)

f1 = f1_score(y_test,log_pred)
print(f'F1 Score of model is', f1*100)

# Multinomial NB

In [ ]:
MNB = MultinomialNB()
MNB.fit(X_train, y_train)
nb_pred = MNB.predict(X_test)

acc = accuracy_score(y_test,nb_pred)
print(f'Accuracy of model is',acc*100)

recall = recall_score(y_test,nb_pred)
print(f'Recall of model is',recall*100)

prec = precision_score(y_test,nb_pred)
print(f'Precision of model is',prec*100)

f1 = f1_score(y_test,nb_pred)
print(f'F1 Score of model is', f1*100)


# Random Forest

In [ ]:
clf = RandomForestClassifier(n_estimators = 100) 
clf.fit(X_train, y_train)
rf_pred = clf.predict(X_test)

acc = accuracy_score(y_test,rf_pred)
print(f'Accuracy of model is',acc*100)

recall = recall_score(y_test,rf_pred)
print(f'Recall of model is',recall*100)

prec = precision_score(y_test,rf_pred)
print(f'Precision of model is',prec*100)

f1 = f1_score(y_test,rf_pred)
print(f'F1 Score of model is', f1*100)

# Working on test data

In [ ]:
test.head()

In [ ]:
test.columns

In [ ]:
test = test.drop(columns=['keyword', 'location'])
test.head()

In [ ]:
test.isna().sum()

In [ ]:
test['text']= test['text'].apply(lambda x: clean_text(x.lower()))
test['text']= test['text'].astype(str)
test['text']= test['text'].apply(transform)


In [ ]:
test['text']

In [ ]:
eval=tfidf_vect.transform(test['text']).toarray()

In [ ]:
import warnings
warnings.simplefilter('ignore')


In [ ]:
final_pred = MNB.predict(eval)

In [ ]:
submission = test[['id']].reset_index(drop=True)
submission['target'] = final_pred.astype('int64')

In [ ]:
submission

In [ ]:
submission.to_csv('submission.csv', index=False)
